# Event-TimeRAF Artifact and Claim Verification

This notebook verifies one completed publication-candidate ZIP or its Kaggle-extracted
directory. It contains no expected metric constants. Every comparison is recomputed
from archived predictions and checked against the manifest-backed tables generated by
the training notebook.

Run `01_event_timeraf_kaggle_pipeline.ipynb` first. Then attach its final output bundle.

## 1. Locate and authenticate the completed run

In [2]:
from io import BytesIO
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import shutil
import sys
import zipfile

import numpy as np
import pandas as pd

# May point to either the publication ZIP or Kaggle's extracted run directory.
FINAL_RUN_SOURCE_OVERRIDE = None

def candidate_archives():
    if FINAL_RUN_SOURCE_OVERRIDE and Path(FINAL_RUN_SOURCE_OVERRIDE).is_file():
        yield Path(FINAL_RUN_SOURCE_OVERRIDE)
    cwd = Path.cwd().resolve()
    yield from sorted(cwd.rglob('event_timeraf_publication_candidate_*.zip'), reverse=True)
    yield cwd / 'event_timeraf_final_run.zip'
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        yield from sorted(kaggle_input.rglob('event_timeraf_publication_candidate_*.zip'), reverse=True)
        yield from sorted(kaggle_input.rglob('event_timeraf_final_run.zip'), reverse=True)

def archive_layout(path):
    try:
        with zipfile.ZipFile(path) as bundle:
            members = [PurePosixPath(name) for name in bundle.namelist()]
            marker = next(
                (item for item in members if item.parts[-3:] == ('outputs', 'logs', 'run_manifest.json')),
                None,
            )
            if marker is None:
                return None
            if any(item.is_absolute() or '..' in item.parts for item in members):
                raise RuntimeError(f'Unsafe archive paths: {path}')
            return PurePosixPath(*marker.parts[:-3])
    except (FileNotFoundError, zipfile.BadZipFile):
        return None

def candidate_run_directories():
    if FINAL_RUN_SOURCE_OVERRIDE and Path(FINAL_RUN_SOURCE_OVERRIDE).is_dir():
        yield Path(FINAL_RUN_SOURCE_OVERRIDE)
    seen = set()
    for search_root in (Path.cwd().resolve(), Path('/kaggle/input')):
        if not search_root.exists():
            continue
        for marker in search_root.rglob('run_manifest.json'):
            if marker.parent.name != 'logs' or marker.parent.parent.name != 'outputs':
                continue
            run_root = marker.parents[2].resolve()
            if run_root in seen:
                continue
            seen.add(run_root)
            if (
                (run_root / 'src' / 'event_timeraf' / 'config.py').exists()
                and (run_root / 'configs' / 'default.yaml').exists()
            ):
                yield run_root

FINAL_RUN_ZIP = None
FINAL_RUN_ROOT = None
PREFIX = None
for candidate in candidate_archives():
    layout = archive_layout(candidate)
    if layout is not None:
        FINAL_RUN_ZIP, PREFIX = candidate.resolve(), layout
        break
if FINAL_RUN_ZIP is None:
    FINAL_RUN_ROOT = next(candidate_run_directories(), None)
if FINAL_RUN_ZIP is None and FINAL_RUN_ROOT is None:
    raise FileNotFoundError(
        'Run notebook 01 to completion, then attach its publication-candidate ZIP '
        'or extracted output Dataset. Set FINAL_RUN_SOURCE_OVERRIDE if discovery fails.'
    )

def member(relative):
    return (PREFIX / PurePosixPath(relative)).as_posix()

def read_bytes(relative):
    if FINAL_RUN_ZIP is not None:
        with zipfile.ZipFile(FINAL_RUN_ZIP) as bundle:
            return bundle.read(member(relative))
    return (FINAL_RUN_ROOT / PurePosixPath(relative)).read_bytes()

def read_json(relative):
    return json.loads(read_bytes(relative).decode('utf-8'))

def read_csv(relative):
    return pd.read_csv(BytesIO(read_bytes(relative)))

def read_parquet(relative):
    return pd.read_parquet(BytesIO(read_bytes(relative)))

def read_npz(relative):
    return np.load(BytesIO(read_bytes(relative)))

manifest = read_json('outputs/logs/run_manifest.json')
integrity_rows = []
for relative, expected in manifest['artifacts'].items():
    payload = read_bytes(relative)
    integrity_rows.append({
        'artifact': relative,
        'bytes_match': len(payload) == expected['bytes'],
        'sha256_match': hashlib.sha256(payload).hexdigest() == expected['sha256'],
    })
integrity = pd.DataFrame(integrity_rows)
display(integrity.groupby(['bytes_match', 'sha256_match']).size().rename('artifact_count').to_frame())
assert integrity[['bytes_match', 'sha256_match']].all().all()
print({
    'source': str(FINAL_RUN_ZIP or FINAL_RUN_ROOT),
    'source_kind': 'zip' if FINAL_RUN_ZIP is not None else 'extracted_directory',
    'prefix': str(PREFIX) if PREFIX is not None else None,
    'run_id': manifest['run_id'],
})

,,artifact_count
bytes_match,sha256_match,
True,True,89


{'source': '/kaggle/input/datasets/sabbir234/firstrun02/event_timeraf_20260817T110915154675Z', 'source_kind': 'extracted_directory', 'prefix': None, 'run_id': '20260817T110915154675Z'}


## 2. Load the archived implementation and result tables

In [3]:
extraction_root = Path('/kaggle/working/event_timeraf_verification_source') if Path('/kaggle/working').exists() else Path('verification_outputs/source')
if extraction_root.exists():
    shutil.rmtree(extraction_root)
extraction_root.mkdir(parents=True)
if FINAL_RUN_ZIP is not None:
    with zipfile.ZipFile(FINAL_RUN_ZIP) as bundle:
        for name in bundle.namelist():
            path = PurePosixPath(name)
            relative = path.parts[len(PREFIX.parts):]
            if relative and relative[0] in {'src', 'configs'}:
                bundle.extract(name, extraction_root)
    project_root = extraction_root.joinpath(*PREFIX.parts)
else:
    shutil.copytree(FINAL_RUN_ROOT / 'src', extraction_root / 'src')
    shutil.copytree(FINAL_RUN_ROOT / 'configs', extraction_root / 'configs')
    project_root = extraction_root
sys.path.insert(0, str(project_root / 'src'))

from event_timeraf.config import load_config
from event_timeraf.evaluation import (
    diebold_mariano_hac, exceedance_metrics, holm_adjust_pvalues,
    horizon_skill_table, interval_metrics, log_scale_metrics, metric_values,
    paired_block_bootstrap_loss_difference, quantile_forecast_metrics,
)

cfg = load_config(project_root / 'configs' / 'default.yaml', project_root)
predictions_long = read_parquet('outputs/predictions/predictions.parquet')
main_results = read_csv('outputs/tables/main_results.csv')
ablation_saved = read_csv('outputs/tables/ablation_results.csv')
exceedance_saved = read_csv('outputs/tables/aqi_exceedance_metrics.csv')
horizon_skill_saved = read_csv('outputs/tables/horizon_skill_vs_climatology.csv')
log_saved = read_csv('outputs/tables/log_scale_metrics.csv')
interval_saved = read_csv('outputs/tables/tsfm_interval_metrics.csv')
quantile_saved = read_csv('outputs/tables/tsfm_quantile_calibration.csv')
probabilistic_saved = read_csv('outputs/tables/tsfm_probabilistic_metrics.csv')
site_level_saved = read_csv('outputs/tables/site_level_sensitivity.csv')
site_design_saved = read_csv('outputs/tables/site_level_design.csv')
site_selection_saved = read_csv('outputs/tables/site_selection_audit.csv')

run_id_sets = [
    set(frame['run_id']) for frame in (
        predictions_long, main_results, ablation_saved, exceedance_saved,
        horizon_skill_saved, log_saved, interval_saved, quantile_saved, probabilistic_saved,
        site_level_saved, site_design_saved,
        site_selection_saved,
    )
]
assert all(values == {manifest['run_id']} for values in run_id_sets)

## 3. Recompute every overall model metric

In [4]:
def model_arrays(model):
    frame = predictions_long.loc[predictions_long['model'] == model].sort_values(['window_id', 'horizon'])
    horizons = int(frame['horizon'].max())
    return (
        frame['actual'].to_numpy().reshape(-1, horizons),
        frame['prediction'].to_numpy().reshape(-1, horizons),
    )

array_cache = {model: model_arrays(model) for model in sorted(predictions_long['model'].unique())}
recomputed_rows = []
for model, (actual, predicted) in array_cache.items():
    recomputed_rows.append({'run_id': manifest['run_id'], 'model': model, **metric_values(actual, predicted)})
recomputed = pd.DataFrame(recomputed_rows)
saved = main_results[['model', 'mse', 'mae', 'rmse', 'mape', 'smape', 'r2']]
comparison = recomputed.merge(saved, on='model', suffixes=('_recomputed', '_saved'), validate='one_to_one')
for metric in ('mse', 'mae', 'rmse', 'mape', 'smape', 'r2'):
    comparison[f'{metric}_absolute_error'] = np.abs(
        comparison[f'{metric}_recomputed'] - comparison[f'{metric}_saved']
    )
metric_error_columns = [name for name in comparison if name.endswith('_absolute_error')]
display(comparison[['model', *metric_error_columns]].sort_values('mse_absolute_error', ascending=False))
assert comparison[metric_error_columns].to_numpy().max() < 1e-5
display(main_results.sort_values('mse'))

,model,mse_absolute_error,mae_absolute_error,rmse_absolute_error,mape_absolute_error,smape_absolute_error,r2_absolute_error
8,C02_calendar_retrieval,7.105427e-15,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.775558e-17
7,C01_ridge_context,7.105427e-15,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
3,B00_dlinear,7.105427e-15,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.551115e-17
22,M10_frozen_chronos,7.105427e-15,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
26,P01_chronos_persistence_fusion,7.105427e-15,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
18,M06_cosine_retrieval,7.105427e-15,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.551115e-17
2,A02_xgb_matched_event_placebo,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
6,C00_hour_month_climatology,0.000000e+00,8.881784e-16,0.000000e+00,5.684342e-14,0.000000e+00,4.163336e-17
1,A01_xgb_random_retrieval,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.551115e-17
0,A00_full_without_events,0.000000e+00,4.440892e-16,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


,run_id,model,subset,event_availability_mode,n_origins,n_points,horizon,mse,mae,rmse,mape,smape,r2
11,20260817T110915154675Z,C05_lightgbm_context,all,retrospective_event_start,7930,190320,overall,35.475716,3.428440,5.956149,169.195937,31.158172,0.378371
9,20260817T110915154675Z,M04_xgb_context,all,retrospective_event_start,7930,190320,overall,35.698334,3.458990,5.974808,174.746955,31.416725,0.374470
19,20260817T110915154675Z,A01_xgb_random_retrieval,all,retrospective_event_start,7930,190320,overall,35.709409,3.458391,5.975735,176.460372,31.405355,0.374276
16,20260817T110915154675Z,C04_xgb_context_event,all,retrospective_event_start,7930,190320,overall,35.747878,3.465761,5.978953,174.318749,31.464246,0.373602
21,20260817T110915154675Z,A00_full_without_events,all,retrospective_event_start,7930,190320,overall,35.773600,3.458569,5.981104,174.325242,31.411392,0.373152
15,20260817T110915154675Z,M07_xgb_cosine,all,retrospective_event_start,7930,190320,overall,35.807077,3.461652,5.983901,179.656625,31.453537,0.372565
20,20260817T110915154675Z,M09_event_timeraf_full,all,retrospective_event_start,7930,190320,overall,35.870537,3.466188,5.989202,171.441533,31.456990,0.371453
17,20260817T110915154675Z,A02_xgb_matched_event_placebo,all,retrospective_event_start,7930,190320,overall,35.870691,3.467460,5.989215,174.325178,31.462993,0.371450
18,20260817T110915154675Z,M08_event_timeraf_no_drift,all,retrospective_event_start,7930,190320,overall,35.978311,3.472462,5.998192,172.697509,31.502427,0.369564
10,20260817T110915154675Z,C01_ridge_context,all,retrospective_event_start,7930,190320,overall,36.917721,3.567546,6.075995,175.151672,32.577319,0.353103


## 4. Recompute bootstrap intervals, DM tests, and Holm adjustment

In [5]:
comparisons = {
    'M04_minus_M03_weather_calendar': ('M04_xgb_context', 'M03_xgb_pm25'),
    'M07_minus_M04_cosine_retrieval': ('M07_xgb_cosine', 'M04_xgb_context'),
    'M08_minus_M07_event_conditioning': ('M08_event_timeraf_no_drift', 'M07_xgb_cosine'),
    'M09_minus_M08_drift_features': ('M09_event_timeraf_full', 'M08_event_timeraf_no_drift'),
    'M09_minus_A00_events': ('M09_event_timeraf_full', 'A00_full_without_events'),
    'M09_minus_M04_full': ('M09_event_timeraf_full', 'M04_xgb_context'),
    'A01_minus_M04_random_control': ('A01_xgb_random_retrieval', 'M04_xgb_context'),
    'M09_minus_A01_random_control': ('M09_event_timeraf_full', 'A01_xgb_random_retrieval'),
    'C04_minus_M04_raw_event_features': ('C04_xgb_context_event', 'M04_xgb_context'),
    'A02_minus_M04_matched_feature_count': ('A02_xgb_matched_event_placebo', 'M04_xgb_context'),
    'C04_minus_A02_event_signal': ('C04_xgb_context_event', 'A02_xgb_matched_event_placebo'),
    'C05_minus_M04_lightgbm': ('C05_lightgbm_context', 'M04_xgb_context'),
    'B00_minus_M04_dlinear': ('B00_dlinear', 'M04_xgb_context'),
    'B01_minus_M04_patchtst': ('B01_patchtst', 'M04_xgb_context'),
    'B02_minus_M04_lstm': ('B02_lstm', 'M04_xgb_context'),
    'M11_minus_M10_retrieval': ('M11_chronos_event_retrieval', 'M10_frozen_chronos'),
    'M11_minus_P00_climatology_placebo': ('M11_chronos_event_retrieval', 'P00_chronos_climatology_fusion'),
    'M11_minus_P01_persistence_placebo': ('M11_chronos_event_retrieval', 'P01_chronos_persistence_fusion'),
    'M12_minus_M04_drift_router': ('M12_validation_drift_router', 'M04_xgb_context'),
}
rows = []
for comparison_name, (model_a, model_b) in comparisons.items():
    actual, prediction_a = array_cache[model_a]
    actual_b, prediction_b = array_cache[model_b]
    assert np.array_equal(actual, actual_b)
    for metric in ('mse', 'mae'):
        interval = paired_block_bootstrap_loss_difference(
            actual, prediction_a, prediction_b, metric,
            cfg.evaluation.bootstrap_block_hours, cfg.evaluation.bootstrap_resamples, cfg.seed,
        )
        dm = diebold_mariano_hac(
            actual, prediction_a, prediction_b, metric, cfg.evaluation.dm_hac_lags
        )
        rows.append({'comparison': comparison_name, 'metric': metric, **interval, **dm})
ablation_recomputed = pd.DataFrame(rows)
for _, indices in ablation_recomputed.groupby('metric').groups.items():
    indices = list(indices)
    ablation_recomputed.loc[indices, 'bootstrap_p_value_holm'] = holm_adjust_pvalues(
        ablation_recomputed.loc[indices, 'bootstrap_p_value'].to_numpy()
    )
    ablation_recomputed.loc[indices, 'dm_p_value_holm'] = holm_adjust_pvalues(
        ablation_recomputed.loc[indices, 'dm_p_value'].to_numpy()
    )
checked_columns = [
    'difference', 'ci_low', 'ci_high', 'bootstrap_p_value', 'dm_statistic',
    'dm_p_value', 'bootstrap_p_value_holm', 'dm_p_value_holm',
]
ablation_check = ablation_recomputed.merge(
    ablation_saved, on=['comparison', 'metric'], suffixes=('_recomputed', '_saved'), validate='one_to_one'
)
errors = []
for column in checked_columns:
    error_name = f'{column}_absolute_error'
    ablation_check[error_name] = np.abs(
        ablation_check[f'{column}_recomputed'] - ablation_check[f'{column}_saved']
    )
    errors.append(error_name)
display(ablation_check[['comparison', 'metric', *errors]])
assert ablation_check[errors].to_numpy().max() < 1e-10

,comparison,metric,difference_absolute_error,ci_low_absolute_error,ci_high_absolute_error,bootstrap_p_value_absolute_error,dm_statistic_absolute_error,dm_p_value_absolute_error,bootstrap_p_value_holm_absolute_error,dm_p_value_holm_absolute_error
0,M04_minus_M03_weather_calendar,mse,0.000000e+00,8.881784e-16,0.000000e+00,1.214306e-17,4.440892e-16,1.734723e-17,2.775558e-17,4.163336e-17
1,M04_minus_M03_weather_calendar,mae,0.000000e+00,8.326673e-17,5.551115e-17,3.252607e-19,0.000000e+00,0.000000e+00,6.071532e-18,0.000000e+00
2,M07_minus_M04_cosine_retrieval,mse,4.163336e-17,5.551115e-17,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
3,M07_minus_M04_cosine_retrieval,mae,7.459311e-17,6.591949e-17,7.285839e-17,0.000000e+00,5.551115e-17,0.000000e+00,0.000000e+00,0.000000e+00
4,M08_minus_M07_event_conditioning,mse,8.326673e-17,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
5,M08_minus_M07_event_conditioning,mae,9.540979e-17,2.602085e-17,4.857226e-17,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
6,M09_minus_M08_drift_features,mse,2.775558e-17,5.551115e-17,5.551115e-17,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
7,M09_minus_M08_drift_features,mae,8.413409e-17,3.469447e-17,3.686287e-17,5.551115e-17,0.000000e+00,2.775558e-17,0.000000e+00,0.000000e+00
8,M09_minus_A00_events,mse,5.551115e-17,5.551115e-17,0.000000e+00,0.000000e+00,5.551115e-17,0.000000e+00,0.000000e+00,0.000000e+00
9,M09_minus_A00_events,mae,4.076600e-17,9.367507e-17,9.020562e-17,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


## 5. Recompute operational, log-scale, and probabilistic metrics

In [6]:
climatology = array_cache['C00_hour_month_climatology'][1]
exceedance_frames, horizon_frames, log_frames = [], [], []
for model, (actual, predicted) in array_cache.items():
    exceedance_frames.append(exceedance_metrics(actual, predicted, cfg.evaluation.aqi_thresholds, model, manifest['run_id']))
    horizon_frames.append(horizon_skill_table(actual, predicted, climatology, model, manifest['run_id']))
    log_frames.append(log_scale_metrics(actual, predicted, model, manifest['run_id']))
operational_recomputed = pd.concat(exceedance_frames, ignore_index=True)
horizon_recomputed = pd.concat(horizon_frames, ignore_index=True)
log_recomputed = pd.concat(log_frames, ignore_index=True)

def maximum_table_error(left, right, keys, columns):
    merged = left.merge(right, on=keys, suffixes=('_recomputed', '_saved'), validate='one_to_one')
    errors = []
    for column in columns:
        difference = np.abs(merged[f'{column}_recomputed'] - merged[f'{column}_saved']).to_numpy()
        finite = difference[np.isfinite(difference)]
        errors.append(float(finite.max()) if len(finite) else 0.0)
    return max(errors)

assert maximum_table_error(
    operational_recomputed, exceedance_saved, ['model', 'threshold_ug_m3'],
    ['precision', 'recall', 'f1', 'critical_success_index', 'auroc', 'average_precision'],
) < 1e-10
assert maximum_table_error(
    horizon_recomputed, horizon_skill_saved, ['model', 'horizon'],
    ['model_mse', 'climatology_mse', 'skill_vs_climatology'],
) < 1e-10
assert maximum_table_error(
    log_recomputed, log_saved, ['model'], ['mse', 'mae', 'rmse', 'r2']
) < 1e-10

tsfm = read_npz('outputs/predictions/tsfm_predictions.npz')
levels = tuple(tsfm['quantile_levels'].tolist())
actual, _ = array_cache['M10_frozen_chronos']
calibration_frames, probability_frames, interval_frames = [], [], []
for model, values in {
    'M10_frozen_chronos': tsfm['test_quantiles'],
    'M11_chronos_event_retrieval': tsfm['fused_test_quantiles'],
}.items():
    calibration, probability = quantile_forecast_metrics(
        actual, values, levels, model, manifest['run_id']
    )
    calibration_frames.append(calibration)
    probability_frames.append(probability)
    interval_frames.append(interval_metrics(
        actual, values[..., levels.index(0.1)], values[..., levels.index(0.9)],
        0.2, model, manifest['run_id'],
    ))
calibration_recomputed = pd.concat(calibration_frames, ignore_index=True)
probability_recomputed = pd.concat(probability_frames, ignore_index=True)
interval_recomputed = pd.concat(interval_frames, ignore_index=True)
assert maximum_table_error(
    calibration_recomputed, quantile_saved, ['model', 'quantile'],
    ['empirical_cdf', 'calibration_error', 'pinball_loss'],
) < 1e-10
assert maximum_table_error(
    probability_recomputed, probabilistic_saved, ['model'],
    ['crps_quantile_approximation', 'mean_absolute_calibration_error'],
) < 1e-10
assert maximum_table_error(
    interval_recomputed, interval_saved, ['model'],
    ['empirical_coverage', 'mean_width', 'winkler_interval_score'],
) < 1e-10
display(probabilistic_saved)
display(exceedance_saved)

,run_id,model,quantile_grid,crps_quantile_approximation,mean_absolute_calibration_error,monotonicity_correction_fraction,n_points
0,20260817T110915154675Z,M10_frozen_chronos,"0.05,0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.45,0...",2.671995,0.011686,0.0,190320
1,20260817T110915154675Z,M11_chronos_event_retrieval,"0.05,0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.45,0...",2.716728,0.059027,0.0,190320


,run_id,model,threshold_ug_m3,aggregation,n_origins,observed_exceedances,forecast_exceedances,tp,fp,fn,tn,precision,recall,f1,critical_success_index,specificity,balanced_accuracy,auroc,average_precision
0,20260817T110915154675Z,C00_hour_month_climatology,35.4,24h_forecast_mean,7930,83,0,0,0,83,7847,NaN,0.000000,NaN,0.000000,1.000000,0.500000,0.664462,0.015769
1,20260817T110915154675Z,C00_hour_month_climatology,55.4,24h_forecast_mean,7930,0,0,0,0,0,7930,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN
2,20260817T110915154675Z,C02_calendar_retrieval,35.4,24h_forecast_mean,7930,83,2,0,2,83,7845,0.000000,0.000000,NaN,0.000000,0.999745,0.499873,0.830426,0.039392
3,20260817T110915154675Z,C02_calendar_retrieval,55.4,24h_forecast_mean,7930,0,0,0,0,0,7930,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN
4,20260817T110915154675Z,C03_event_conditioned_retrieval,35.4,24h_forecast_mean,7930,83,17,5,12,78,7835,0.294118,0.060241,0.100000,0.052632,0.998471,0.529356,0.848841,0.180146
5,20260817T110915154675Z,C03_event_conditioned_retrieval,55.4,24h_forecast_mean,7930,0,0,0,0,0,7930,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN
6,20260817T110915154675Z,M00_persistence,35.4,24h_forecast_mean,7930,83,113,30,83,53,7764,0.265487,0.361446,0.306122,0.180723,0.989423,0.675434,0.869687,0.190313
7,20260817T110915154675Z,M00_persistence,55.4,24h_forecast_mean,7930,0,31,0,31,0,7899,0.000000,NaN,NaN,0.000000,0.996091,NaN,NaN,NaN
8,20260817T110915154675Z,M01_daily_seasonal,35.4,24h_forecast_mean,7930,83,85,17,68,66,7779,0.200000,0.204819,0.202381,0.112583,0.991334,0.598077,0.839415,0.206795
9,20260817T110915154675Z,M01_daily_seasonal,55.4,24h_forecast_mean,7930,0,0,0,0,0,7930,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN


## 6. Recompute the site-level sensitivity arm

In [7]:
site_arrays = read_npz('outputs/predictions/site_level_predictions.npz')
site_rows = []
for site_id in site_design_saved['site_id']:
    site_key = str(site_id).replace('-', '_')
    actual = site_arrays[f'{site_key}_actual']
    climatology = site_arrays[f'{site_key}_climatology']
    event_mask = site_arrays[f'{site_key}_event_mask'].astype(bool)
    for subset, mask in {
        'all': np.ones(len(actual), dtype=bool),
        'event': event_mask,
        'non_event': ~event_mask,
    }.items():
        eligible = int(mask.sum()) >= cfg.evaluation.minimum_subset_origins
        climatology_mse = metric_values(actual[mask], climatology[mask])['mse'] if eligible else np.nan
        for model, suffix in {
            'S_M04_xgb_context': 'M04',
            'S_M08_event_retrieval': 'M08',
        }.items():
            predicted = site_arrays[f'{site_key}_{suffix}']
            scores = metric_values(actual[mask], predicted[mask]) if eligible else {
                'mse': np.nan, 'mae': np.nan, 'rmse': np.nan, 'r2': np.nan
            }
            site_rows.append({
                'site_id': site_id, 'subset': subset, 'model': model,
                **scores,
                'skill_vs_site_climatology': (
                    1 - scores['mse'] / climatology_mse
                    if eligible and climatology_mse > 0 else np.nan
                ),
            })
site_recomputed = pd.DataFrame(site_rows)
assert maximum_table_error(
    site_recomputed, site_level_saved, ['site_id', 'subset', 'model'],
    ['mse', 'mae', 'rmse', 'r2', 'skill_vs_site_climatology'],
) < 1e-10
display(site_level_saved)

,run_id,site_id,subset,model,n_origins,eligible_for_metrics,mse,mae,rmse,mape,smape,r2,skill_vs_site_climatology
0,20260817T110915154675Z,06-037-1103,all,S_M04_xgb_context,6339,True,111.468093,5.067968,10.557845,59.921489,33.892284,0.191415,0.240983
1,20260817T110915154675Z,06-037-1103,all,S_M08_event_retrieval,6339,True,113.119934,5.054412,10.635786,60.675307,33.794549,0.179433,0.229735
2,20260817T110915154675Z,06-037-1103,event,S_M04_xgb_context,422,True,296.272936,8.219362,17.212581,313.703747,42.651775,0.126935,0.258632
3,20260817T110915154675Z,06-037-1103,event,S_M08_event_retrieval,422,True,297.192436,8.322077,17.239270,326.630393,43.385479,0.124225,0.256331
4,20260817T110915154675Z,06-037-1103,non_event,S_M04_xgb_context,5917,True,98.287826,4.843211,9.914022,41.821758,33.267557,0.200356,0.237079
5,20260817T110915154675Z,06-037-1103,non_event,S_M08_event_retrieval,5917,True,99.991897,4.821362,9.999595,41.707410,33.110524,0.186492,0.223851
6,20260817T110915154675Z,06-037-4009,all,S_M04_xgb_context,5634,True,64.431083,4.755497,8.026897,557.196060,41.852853,0.219407,0.159514
7,20260817T110915154675Z,06-037-4009,all,S_M08_event_retrieval,5634,True,63.594783,4.732951,7.974634,579.494929,41.678920,0.229539,0.170423
8,20260817T110915154675Z,06-037-4009,event,S_M04_xgb_context,354,True,149.004819,7.638273,12.206753,1711.256040,50.786515,0.118600,0.165483
9,20260817T110915154675Z,06-037-4009,event,S_M08_event_retrieval,354,True,146.860976,7.605741,12.118621,1391.676909,51.023557,0.131282,0.177490


## 7. Final reproducibility gate

In [8]:
required_tables = {
    'window_origin_attrition.csv', 'kb_stride_sensitivity.csv',
    'kb_stride_model_sensitivity.csv', 'event_weight_sensitivity.csv',
    'event_weight_model_sensitivity.csv', 'event_category_candidate_composition.csv',
    'validation_group_faithfulness.csv',
    'drift_leaf_occupancy.csv',
    'neural_baseline_training.csv', 'feature_count_control_design.csv',
    'tsfm_placebo_fusion_validation.csv',
    'tsfm_quantile_calibration.csv', 'tsfm_probabilistic_metrics.csv',
    'site_level_sensitivity.csv', 'site_level_design.csv',
    'site_selection_audit.csv',
}
required_figures = {
    'mae_by_horizon.png', 'mse_by_horizon.png', 'forecast_case.png',
    'retrieval_diagnostics.png', 'drift_scores.png',
}
if FINAL_RUN_ZIP is not None:
    with zipfile.ZipFile(FINAL_RUN_ZIP) as bundle:
        names = {PurePosixPath(name).name for name in bundle.namelist()}
else:
    names = {path.name for path in FINAL_RUN_ROOT.rglob('*') if path.is_file()}
missing = sorted((required_tables | required_figures) - names)
assert not missing, f'Missing publication artifacts: {missing}'
options = manifest['run_options']
gate_rows = [
    {'gate': 'manifest hashes', 'passed': bool(integrity['sha256_match'].all())},
    {'gate': '1/6/24-hour KB sweep', 'passed': set(options.get('kb_stride_values', [])) == {1, 6, 24}},
    {'gate': 'journal baselines', 'passed': bool(options.get('journal_baselines_completed'))},
    {'gate': 'learned event-weight sweep', 'passed': bool(options.get('event_weight_model_sweep_completed'))},
    {'gate': 'matched feature-count control', 'passed': bool(options.get('matched_feature_count_control'))},
    {'gate': 'three-monitor site arm', 'passed': bool(options.get('site_level_sensitivity_completed'))},
    {'gate': 'Holm-adjusted DM inference', 'passed': bool(options.get('holm_adjustment'))},
    {'gate': 'probabilistic quantile grid', 'passed': len(options.get('probabilistic_quantiles', [])) >= 19},
]
gates = pd.DataFrame(gate_rows)
display(gates)
assert gates['passed'].all()
print('Verification complete for run', manifest['run_id'])

,gate,passed
0,manifest hashes,True
1,1/6/24-hour KB sweep,True
2,journal baselines,True
3,learned event-weight sweep,True
4,matched feature-count control,True
5,three-monitor site arm,True
6,Holm-adjusted DM inference,True
7,probabilistic quantile grid,True


Verification complete for run 20260817T110915154675Z
